In [1]:
import os
import json
import numpy as np

In [2]:
# base directory that contains ADFTD-RS and ADFTD-PS
base_dir = "Processed/L400"

# source folders
rs_dir = os.path.join(base_dir, "ADFTD-RS")
ps_dir = os.path.join(base_dir, "ADFTD-PS")

# output folder (will be created)
out_dir = os.path.join(base_dir, "ADFTD")
os.makedirs(out_dir, exist_ok=True)

print("RS:", rs_dir)
print("PS:", ps_dir)
print("OUT:", out_dir)

RS: Processed/L400\ADFTD-RS
PS: Processed/L400\ADFTD-PS
OUT: Processed/L400\ADFTD


In [3]:
# ------ load meta.json from each dataset (RS, PS) ------
def load_meta(folder):
    """Load meta.json from a folder."""
    meta_path = os.path.join(folder, "meta.json")
    with open(meta_path, "r") as f:
        meta = json.load(f)
    return meta

meta_rs = load_meta(rs_dir)
meta_ps = load_meta(ps_dir)

print("RS meta loaded")
print("PS meta loaded")

RS meta loaded
PS meta loaded


In [4]:
# ------ verify T and C match ------
T_rs, C_rs = meta_rs["T"], meta_rs["C"]
T_ps, C_ps = meta_ps["T"], meta_ps["C"]

if T_rs != T_ps or C_rs != C_ps:
    raise ValueError(f"T/C mismatch: RS(T={T_rs}, C={C_rs}), PS(T={T_ps}, C={C_ps})")

T, C = T_rs, C_rs
print(f"Verified: T={T}, C={C}")

Verified: T=400, C=19


In [5]:
# ------ compute total N and create new memmap files ------
N_rs = meta_rs["N"]
N_ps = meta_ps["N"]
N_total = N_rs + N_ps

print(f"N_rs={N_rs}, N_ps={N_ps}, N_total={N_total}")

X_out_path = os.path.join(out_dir, "X.dat")
y_out_path = os.path.join(out_dir, "y.dat")

# allocate merged memmap
X_merged = np.memmap(X_out_path, dtype="float32", mode="w+", shape=(N_total, T, C))
y_merged = np.memmap(y_out_path, dtype="float32", mode="w+", shape=(N_total, 3))

N_rs=121825, N_ps=45258, N_total=167083


In [6]:
# ------ helper function to copy memmap blocks ------
def copy_block(src_folder, meta, X_dst, y_dst, offset):
    """
    Copy X.dat and y.dat from a source dataset into the merged memmap.
    offset: starting row index in the merged dataset.
    """
    N = meta["N"]
    T = meta["T"]
    C = meta["C"]

    X_path = os.path.join(src_folder, "X.dat")
    y_path = os.path.join(src_folder, "y.dat")

    print(f"Copying from: {src_folder}")
    print(f"  offset={offset}, N={N}")

    # open source memmaps
    X_src = np.memmap(X_path, dtype="float32", mode="r", shape=(N, T, C))
    y_src = np.memmap(y_path, dtype="float32", mode="r", shape=(N, 3))

    # copy blocks
    X_dst[offset:offset + N] = X_src
    y_dst[offset:offset + N] = y_src

    # release memmap views
    del X_src, y_src

    # return new offset
    return offset + N

In [7]:
# ------ perform merging ------
cur = 0

print("\n--- Copy RS ---")
cur = copy_block(rs_dir, meta_rs, X_merged, y_merged, cur)

print("\n--- Copy PS ---")
cur = copy_block(ps_dir, meta_ps, X_merged, y_merged, cur)

print("\nFinished merging. Final cur =", cur)


--- Copy RS ---
Copying from: Processed/L400\ADFTD-RS
  offset=0, N=121825

--- Copy PS ---
Copying from: Processed/L400\ADFTD-PS
  offset=121825, N=45258

Finished merging. Final cur = 167083


In [8]:
# ------ write merged meta.json ------
meta_out = dict(meta_rs)   # copy RS meta as template
meta_out["N"] = int(N_total)
meta_out["X_path"] = X_out_path
meta_out["y_path"] = y_out_path
meta_out["sources"] = {
    "RS": {"folder": rs_dir, "N": int(N_rs)},
    "PS": {"folder": ps_dir, "N": int(N_ps)}
}

meta_out_path = os.path.join(out_dir, "meta.json")
with open(meta_out_path, "w") as f:
    json.dump(meta_out, f, indent=2)

print("Saved merged meta.json:", meta_out_path)

Saved merged meta.json: Processed/L400\ADFTD\meta.json


## Load and check the processed data

In [9]:
# ------ verify merged data ------
# load meta information
meta_path = meta_out_path  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]
X_path = meta["X_path"]
y_path = meta["y_path"]

print("Meta:")
print("  N =", N)
print("  T =", T)
print("  C =", C)
print("  X_path =", X_path)
print("  y_path =", y_path)
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path, dtype='float32', mode='r', shape=(N, 3))

# subject_id is stored in y[:, 1]
subject_ids = np.unique(y_mm[:, 1]).astype(int)

for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 1] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 3)       # logical y shape for this subject

    # get sampling rates for all segments of this subject
    fs_subject = y_mm[idx, 2]  # shape (n_seg,)


    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

# option: delete memmap views
del X_mm, y_mm

Meta:
  N = 167083
  T = 400
  C = 19
  X_path = Processed/L400\ADFTD\X.dat
  y_path = Processed/L400\ADFTD\y.dat
-------------------------------------
Subject ID 001: X shape=(1570, 400, 19), y shape=(1570, 3)
Subject ID 002: X shape=(2057, 400, 19), y shape=(2057, 3)
Subject ID 003: X shape=(1008, 400, 19), y shape=(1008, 3)
Subject ID 004: X shape=(1814, 400, 19), y shape=(1814, 3)
Subject ID 005: X shape=(1912, 400, 19), y shape=(1912, 3)
Subject ID 006: X shape=(1780, 400, 19), y shape=(1780, 3)
Subject ID 007: X shape=(2079, 400, 19), y shape=(2079, 3)
Subject ID 008: X shape=(2047, 400, 19), y shape=(2047, 3)
Subject ID 009: X shape=(1780, 400, 19), y shape=(1780, 3)
Subject ID 010: X shape=(2754, 400, 19), y shape=(2754, 3)
Subject ID 011: X shape=(1847, 400, 19), y shape=(1847, 3)
Subject ID 012: X shape=(2126, 400, 19), y shape=(2126, 3)
Subject ID 013: X shape=(2265, 400, 19), y shape=(2265, 3)
Subject ID 014: X shape=(2496, 400, 19), y shape=(2496, 3)
Subject ID 015: X shap